# TurnWave on eot-bench — measured against published models

Runs LiveKit's **own** harness rather than our metrics, because eot-bench's
definitions are subtle: a false cutoff is counted per mid-turn pause (not per
row), the 300/600 ms columns are budgets on *mean latency* rather than wait
thresholds, and latency runs from the start of the final silence and includes
the policy's own action delay. Working from the column headings would produce
numbers that look comparable and are not.

The published English leaderboard, which this fills the last row of:

| model | false cutoffs @300 ms | @600 ms | latency @5% cutoff |
|---|---|---|---|
| VAD baseline | 55.6% | 21.7% | 1600 ms |
| SmartTurn v3.2 | 35.2% | 14.8% | 1051 ms |
| LiveKit Turn Detector v1 | 9.9% | 4.5% | 543 ms |
| **TurnWave** | ? | ? | ? |

**Upload `turnwave_phase4.zip` first** (Files pane -> upload), or re-run the
Phase 4 notebook in this session. A GPU is not required — inference only.

~45 min including a 1.9 GB dataset download.

In [ ]:
# 1. Repo, deps, and the harness.
import os, subprocess, sys

if not os.path.isdir('/content/turnwave'):
    !git clone -q https://github.com/Nikhils-G/turnwave.git /content/turnwave
%cd /content/turnwave
!git pull -q
install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
                          '--no-deps', 'sentencepiece', 'datasets', 'soundfile',
                          'onnx', 'onnxruntime', 'onnxscript'])
assert install.returncode == 0, 'install failed'
!pip install -q git+https://github.com/livekit/eot-bench
print('harness installed')

In [ ]:
# 2. Unpack the trained models and confirm the adapter loads them.
if os.path.exists('/content/turnwave_phase4.zip'):
    !unzip -qo /content/turnwave_phase4.zip -d /content/turnwave
assert os.path.exists('checkpoints/onnx/fusion_eot.onnx'), \
    'no exported model - upload turnwave_phase4.zip to /content first'

os.environ['TURNWAVE_ONNX'] = 'checkpoints/onnx/fusion_eot.onnx'
os.environ['TURNWAVE_TOKENIZER'] = 'checkpoints/tokenizer/spm.model'
from turnwave.eot_bench_adapter import TurnWaveAdapter
a = TurnWaveAdapter()
print('adapter OK | score_point', a.score_point, '| needs audio', a.detector.needs_audio,
      '| needs text', a.detector.needs_text)

In [ ]:
# 3. Score the fused model on real human-to-agent audio (~30 min, 1.9 GB).
!eot-harness predict --path livekit/eot-bench-data --name all --split validation \
    --adapter turnwave.eot_bench_adapter:TurnWaveAdapter --output-dir output

In [ ]:
# 4. Each branch alone, so the ablation is reproduced on REAL conversational
# audio. Our own test set is isolated utterances; this checks whether fusion's
# advantage survives the domain shift, which is worth as much as the headline.
os.environ['TURNWAVE_ONNX'] = 'checkpoints/onnx/audio_eot.onnx'
!eot-harness predict --path livekit/eot-bench-data --name all --split validation \
    --adapter turnwave.eot_bench_adapter:TurnWaveAudioOnlyAdapter --output-dir output

os.environ['TURNWAVE_ONNX'] = 'checkpoints/onnx/text_eot.int8.onnx'
!eot-harness predict --path livekit/eot-bench-data --name all --split validation \
    --adapter turnwave.eot_bench_adapter:TurnWaveTextOnlyAdapter --output-dir output

In [ ]:
# 5. Metrics for every run, then the comparison table.
import glob
for predictions in glob.glob('output/**/predictions.parquet', recursive=True):
    !eot-harness compute-metrics --predictions "{predictions}" \
        --output-dir "{os.path.dirname(predictions)}/metrics"

run_root = 'output/livekit__eot-bench-data__validation__min_silence_100ms/en'
!eot-harness compare-models {run_root}

In [ ]:
# 6. Show the report, then save it.
report = f'{run_root}/comparison/report.md'
if os.path.exists(report):
    print(open(report).read())
!zip -qr turnwave_eot_bench.zip output
from google.colab import files
files.download('turnwave_eot_bench.zip')